# 02 · The centralized baseline

Every later phase is reported as a gap to this model, so it has to be both
strong and honestly measured. This notebook answers three questions:

1. **How close is it to the published PTB-XL result?** The reference is
   `resnet1d_wang` at 0.930 macro AUROC (Strodthoff et al., 2021).
2. **What did the training recipe buy?** Random crops and a cosine schedule,
   measured against the plain recipe on the same test fold.
3. **Which superclasses are hard?** Macro AUROC averages five classes that are
   not equally easy.

Training happens in `scripts/train_centralized.py`; this notebook only reads
what it wrote to `results/tables/`. Prerequisite:

```bash
uv run python scripts/train_centralized.py --config centralized.yaml
uv run python scripts/train_centralized.py --config centralized_plain.yaml
```

In [ ]:
%matplotlib inline
import pandas as pd

from fedecg.paths import TABLES_DIR
from fedecg.viz import plot_curves

experiments = pd.read_csv(TABLES_DIR / "experiments.csv")
baseline = experiments.set_index("run").loc["centralized"]


def history(run: str) -> pd.DataFrame:
    return pd.read_csv(TABLES_DIR / f"{run}_history.csv")


def with_gap(rows: pd.DataFrame) -> pd.DataFrame:
    """Add the AUROC gap to the centralized baseline, the number each phase is about."""
    rows = rows.assign(gap=rows["macro_auroc"] - baseline["macro_auroc"])
    return rows.set_index("setting")

## The recipe, and how it was chosen

Three changes, the first two borrowed from the PTB-XL benchmark, all in
`fedecg.training.loop`:

- **Random-crop training.** Each time a record is drawn, a random 2.5 s window
  (250 samples) is cut from its 10 s. The network sees a different view of
  every record each epoch, which regularizes it, and each step is about four
  times cheaper.
- **Cosine learning-rate schedule.** A linear warmup over the first 5% of steps,
  then a half-cosine decay to zero.
- **Scoring the whole record.** The ResNet ends in global average and max
  pooling, so a model trained on 2.5 s crops can score the full 10 s at once.

The recipe was picked **on the validation fold only**, in a one-off sweep
(same seed, same model, early stopping with patience 10; unless stated,
validation scores the whole 10 s record):

| Variant | Val macro AUROC |
|---|---|
| Full-length records, constant LR (the first baseline) | 0.9139 |
| Full-length records, cosine LR | 0.9138 |
| 250-sample crops, cosine LR, 30 epochs | 0.9198 |
| 250-sample crops, cosine LR, LR 3e-3, 30 epochs | 0.9178 |
| 500-sample crops, cosine LR, 30 epochs | 0.9193 |
| **250-sample crops, cosine LR, up to 50 epochs** | **0.9204** |
| 250-sample crops, 30 epochs, 2.5 s windows averaged at evaluation | 0.9193 |
| 250-sample crops, 30 epochs, 2.5 s windows max-pooled at evaluation | 0.9172 |

The schedule alone did nothing; the crops did the work. Averaging overlapping
windows at evaluation, as the benchmark does, was no better than scoring the
whole record and costs seven forward passes instead of one.

## 1 and 2: the test fold

The test fold is scored once per run, with per-class thresholds tuned on
validation. The plain recipe is rerun rather than quoted, so both rows come
from the same code and the same test records.

In [ ]:
published = pd.DataFrame(
    {"setting": ["Published resnet1d_wang (Strodthoff et al., 2021)"], "macro_auroc": [0.930]}
)
phase3 = experiments[experiments["phase"] == 3]
table = pd.concat(
    [phase3[["setting", "macro_auroc", "macro_f1", "best_step", "seconds"]], published]
)
table.set_index("setting").round(4)

In [ ]:
fig = plot_curves(
    {row.setting: history(row.run) for row in phase3.itertuples()},
    step_label="epoch",
    reference=0.930,
    reference_label="published resnet1d_wang (test)",
)

Curves are validation AUROC per epoch; the horizontal line is a *test* number
from the paper, drawn for scale only.

## 3. Per-class performance

In [ ]:
pd.read_csv(TABLES_DIR / "centralized_test_metrics.csv", index_col="superclass")

## What this means

- **The baseline is 0.916 macro AUROC, 0.014 below the published 0.930.** The
  published model differs in configuration and training pipeline; closing the rest of the gap is not needed for this project, whose
  numbers are all *gaps to this baseline*.
- **The recipe is worth +0.007 on test** (0.916 against 0.909), matching the
  +0.0065 it showed on validation. The plain rerun reproduced the first
  baseline's 0.909 exactly, so the pipeline is deterministic.
- **Seed noise is small.** Retraining the baseline with seeds 42, 43 and 44 gives
  0.9156, 0.9149 and 0.9170: a spread of 0.002. Gaps between runs smaller than
  about 0.003 should not be read as real; every other run in this project uses
  seed 42 only.
- **HYP is the hard class** (0.850 AUROC, 0.52 F1), as in the PTB-XL benchmark.
  It is the rarest superclass (12% of records) and only 20% of HYP records
  carry no other label, so
  the model has few clean examples to learn it from.